In [ ]:
# ==============================================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN (Bỏ dấu # ở dòng dưới nếu chạy trên Colab/Jupyter chưa có sẵn)
# ==============================================================================
!pip3 install transformers torch pandas tqdm

In [ ]:


# ==============================================================================
# CELL 2: KHỞI TẠO MÔI TRƯỜNG & IMPORT
# ==============================================================================
import pandas as pd
import torch
import os
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm

# Tự động nhận diện GPU (Khuyến nghị dùng GPU T4 trên Colab để dịch nhanh hơn)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Hệ thống đang chạy trên nền tảng: {device.upper()}")


In [ ]:

# ==============================================================================
# CELL 3: TẢI MÔ HÌNH AI DỊCH THUẬT (HELSINKI-NLP)
# ==============================================================================
print("[INFO] Đang tải mô hình dịch Anh-Việt (Helsinki-NLP/opus-mt-en-vi)...")
model_name = "Helsinki-NLP/opus-mt-en-vi"

# Load Tokenizer và Model, sau đó đẩy model lên GPU (nếu có)
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)
print("[INFO] Tải mô hình thành công!\n")

def translate_en_to_vi(text):
    """Hàm xử lý dịch thuật bằng AI"""
    if not text or len(str(text).strip()) == 0:
        return ""
        
    # Tiền xử lý (cắt độ dài tối đa 512 tokens để tránh lỗi tràn RAM)
    inputs = tokenizer(str(text), return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    
    with torch.no_grad(): # Tắt tính toán gradient để tối ưu hiệu suất infer
        translated = model.generate(**inputs)
        
    return tokenizer.decode(translated[0], skip_special_tokens=True)


In [ ]:

# ==============================================================================
# CELL 4: ĐỌC DỮ LIỆU & KIỂM TRA TRẠNG THÁI KHUYẾT
# ==============================================================================
INPUT_FILE = "tmdb_latest_movies.csv" # Đổi tên file này đúng với tên file của em
OUTPUT_FILE = "movies_data_fully_translated.csv"

# Kiểm tra file tồn tại
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Không tìm thấy file {INPUT_FILE}. Vui lòng upload lên Colab!")

df = pd.read_csv(INPUT_FILE)

# Chuẩn hóa dữ liệu null về chuỗi rỗng
df['overview_vi'] = df['overview_vi'].fillna('')
df['overview'] = df['overview'].fillna('')

# Khởi tạo cột cờ hiệu (Data Provenance) nếu chưa có
if 'is_ai_translated' not in df.columns:
    df['is_ai_translated'] = False

# Tìm các dòng có overview tiếng Anh nhưng chưa có overview tiếng Việt
mask = (df['overview_vi'].str.strip() == '') & (df['overview'].str.strip() != '')
indexes_to_translate = df[mask].index.tolist()

print(f"[STAT] Tổng số phim trong hệ thống: {len(df)}")
print(f"[STAT] Số lượng phim bị khuyết overview_vi cần dịch: {len(indexes_to_translate)}\n")


In [ ]:

# ==============================================================================
# CELL 5: THỰC THI DỊCH TỰ ĐỘNG BẰNG VÒNG LẶP
# ==============================================================================
if len(indexes_to_translate) > 0:
    # Lướt qua từng index cần dịch với thanh tiến trình trực quan
    for idx in tqdm(indexes_to_translate, desc="Đang dịch tự động"):
        en_text = df.at[idx, 'overview']
        
        try:
            # Gọi mô hình AI
            vi_text = translate_en_to_vi(en_text)
            
            # Cập nhật kết quả vào DataFrame
            df.at[idx, 'overview_vi'] = vi_text
            df.at[idx, 'is_ai_translated'] = True
            
        except Exception as e:
            print(f"\n[ERROR] Lỗi tại dòng {idx}: {e}")
            continue
            
    print("\n[INFO] Quá trình dịch hoàn tất!")
else:
    print("[INFO] Dữ liệu đã đầy đủ, không có phim nào cần dịch thêm.")


In [ ]:

# ==============================================================================
# CELL 6: KIỂM TRA VÀ XUẤT FILE KẾT QUẢ
# ==============================================================================
# Lưu dữ liệu hoàn chỉnh
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"[SUCCESS] Đã lưu dữ liệu hoàn thiện tại: {OUTPUT_FILE}")

# In ra 3 mẫu AI vừa dịch để kiểm chứng chất lượng
if len(indexes_to_translate) > 0:
    print("\n--- MỘT SỐ KẾT QUẢ AI VỪA DỊCH ---")
    sample_df = df[df['is_ai_translated'] == True].head(3)
    for _, row in sample_df.iterrows():
        print(f"🎬 Phim: {row.get('title', 'Unknown')}")
        print(f"🇬🇧 Bản gốc: {row['overview'][:100]}...")
        print(f"🇻🇳 AI Dịch: {row['overview_vi'][:100]}...\n")